# Exploratory Data Analysis (EDA)

This notebook contains exploratory analysis of the sensor data.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add src to path to import config
sys.path.append(os.path.join(os.path.abspath(''), '..', 'src'))
import config

# Set plot style
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11, 'xtick.labelsize': 10, 'ytick.labelsize': 10, 'figure.dpi': 300, 'savefig.dpi': 300})

df = pd.read_csv(config.FEATURE_DATA_PATH)


## 1. Distribution plots for key sensor variables

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sns.histplot(df['Temperature'], bins=50, ax=axes[0, 0], kde=True)
axes[0, 0].set_title('Temperature Distribution')

sns.histplot(df['Humidity'], bins=50, ax=axes[0, 1], kde=True)
axes[0, 1].set_title('Humidity Distribution')

sns.histplot(df['pH'], bins=50, ax=axes[1, 0], kde=True)
axes[1, 0].set_title('pH Distribution')

sns.histplot(df['Soil_Moisture'], bins=50, ax=axes[1, 1], kde=True)
axes[1, 1].set_title('Soil Moisture Distribution')

plt.tight_layout()
plt.savefig(os.path.join(config.PLOTS_DIR, 'sensor_distributions.png'))
plt.show()

## 2. Correlation heatmap

In [ ]:
numeric_df = df.select_dtypes(include=['float64', 'int64'])
plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(os.path.join(config.PLOTS_DIR, 'correlation_heatmap.png'))
plt.show()

## 3. MOI distribution by Soil Type

In [ ]:
df_unified = pd.read_csv(config.UNIFIED_DATA_PATH)
df_unified['Temperature'] = df_unified['Temperature'].replace(0, 0.001)
df_unified['Soil_Moisture'] = df_unified['Soil_Moisture'].replace(0, 0.001)
df_unified['MOI'] = (df_unified['Humidity'] / df_unified['Soil_Moisture']) / df_unified['Temperature']

plt.figure(figsize=(12, 6))
sns.boxplot(x='Soil_Type', y='MOI', data=df_unified)
plt.title('MOI Distribution by Soil Type')
plt.xticks(rotation=45)
plt.ylim(0, df_unified['MOI'].quantile(0.95))
plt.tight_layout()
plt.savefig(os.path.join(config.PLOTS_DIR, 'moi_by_soil_type.png'))
plt.show()

## 4. Time-series line plot of Δ pH showing flagged drift events

In [ ]:
plt.figure(figsize=(14, 6))
sample_df = df.head(1000).reset_index()
plt.plot(sample_df.index, sample_df['pH_delta'], label='Δ pH', alpha=0.7)

flagged = sample_df[sample_df['Intervention_Required'] == 1]
plt.scatter(flagged.index, flagged['pH_delta'], color='red', label='Flagged Drift Event', zorder=5)

plt.title('Time-Series of Δ pH with Flagged Drift Events (Sample)')
plt.xlabel('Time (Index)')
plt.ylabel('Δ pH')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(config.PLOTS_DIR, 'delta_ph_timeseries.png'))
plt.show()

## 5. Pairplot of key features

In [ ]:
sns.pairplot(df.sample(500)[['Temperature', 'Humidity', 'pH', 'Soil_Moisture', 'Intervention_Required']], hue='Intervention_Required')
plt.savefig(os.path.join(config.PLOTS_DIR, 'eda_pairplot.png'))
plt.show()